# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/balabhadra3141/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup

In [1]:
import os
import duckdb
import pandas as pd
import numpy as np

from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN was not found. "
        "Add your Hugging Face READ token as a Colab Secret named 'HF_TOKEN'."
    )

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
    """
)

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"
DIM_CONTENT = f"{REL}/dim_content.parquet"

FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

print("Connected to FlyRank warehouse.")
print("Feature window: February 2026")
print("Outcome window: March 2026")

Connected to FlyRank warehouse.
Feature window: February 2026
Outcome window: March 2026


## 1. Method choice and why

I will use **Logistic Regression** as the first capstone model.

The target is whether a content item receives zero GSC clicks in March 2026, using February 2026 information available at the decision point.

Logistic Regression fits this lane because the task is a binary classification problem and the model produces a probability that can be used to rank content for review.

I am starting with a simple, interpretable model rather than a more complex model. This makes it easier to compare the learned relationships with the Week-4 rule and inspect errors.

The model will use only decision-time features from February. March information is used only to create the held-out outcome label.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

I will use a **client-grouped split** so that content from the same client does not appear in both training and test sets.

The split is 80% of clients for training and 20% of clients for testing, using a fixed random seed.

This is more conservative than randomly splitting individual content rows because pages from the same client can share characteristics. Keeping clients separated gives a cleaner test of whether the learned relationship transfers to clients not seen during training.

The February features are measured before the March outcome window. The March zero-click label is used only for evaluation.

In [2]:
# February decision-time features + March outcome.

model_df = con.sql(f"""
WITH feb AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS impressions_feb,

        SUM(gsc_clicks) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS clicks_feb,

        CASE
            WHEN SUM(gsc_impressions) FILTER (
                WHERE gsc_data_available IS TRUE
            ) > 0
            THEN
                CAST(
                    SUM(gsc_sum_position) FILTER (
                        WHERE gsc_data_available IS TRUE
                    ) AS DOUBLE
                )
                /
                SUM(gsc_impressions) FILTER (
                    WHERE gsc_data_available IS TRUE
                )
            ELSE NULL
        END AS avg_position_feb

    FROM {FEB}

    GROUP BY
        client_hash_id,
        content_hash_id
),

mar AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_clicks) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS clicks_mar,

        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS available_days_mar

    FROM {MAR}

    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    f.client_hash_id,
    f.content_hash_id,
    f.impressions_feb,
    f.clicks_feb,
    f.avg_position_feb,
    m.clicks_mar,
    m.available_days_mar,

    CASE
        WHEN m.available_days_mar > 0
             AND COALESCE(m.clicks_mar, 0) = 0
        THEN 1
        ELSE 0
    END AS march_zero_click

FROM feb f

JOIN '{DIM_CONTENT}' c
    ON f.content_hash_id = c.content_hash_id

JOIN mar m
    ON f.client_hash_id = m.client_hash_id
    AND f.content_hash_id = m.content_hash_id

WHERE
    f.impressions_feb >= 100
    AND f.clicks_feb >= 3
    AND f.avg_position_feb IS NOT NULL
    AND c.is_published IS TRUE
""").df()

print("Modeling rows:", len(model_df))
display(model_df.head())

Modeling rows: 29362


,client_hash_id,content_hash_id,impressions_feb,clicks_feb,avg_position_feb,clicks_mar,available_days_mar,march_zero_click
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,601.0,3.0,4.633943,2.0,31,0
1,client_73cda7b4e4f265ea,content_05434271b257bb68,1567.0,7.0,5.641353,6.0,31,0
2,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2176.0,4.0,3.807445,16.0,31,0
3,client_73cda7b4e4f265ea,content_712c365258cee05c,5368.0,21.0,3.818927,23.0,31,0
4,client_73cda7b4e4f265ea,content_8935ed68eca88b01,3248.0,7.0,10.112069,10.0,31,0


In [5]:
# check target
print("Target distribution:")
display(
    model_df["march_zero_click"]
    .value_counts()
    .rename_axis("march_zero_click")
    .reset_index(name="n")
)

print("\nTarget rate:")
print(model_df["march_zero_click"].mean())

Target distribution:


,march_zero_click,n
0,0,28203
1,1,1159



Target rate:
0.03947278795722362


In [6]:
# create client split
from sklearn.model_selection import train_test_split

clients = model_df["client_hash_id"].dropna().unique()

train_clients, test_clients = train_test_split(
    clients,
    test_size=0.20,
    random_state=42
)

train_df = model_df[
    model_df["client_hash_id"].isin(train_clients)
].copy()

test_df = model_df[
    model_df["client_hash_id"].isin(test_clients)
].copy()

print("Training clients:", len(train_clients))
print("Testing clients:", len(test_clients))

print("\nTraining rows:", len(train_df))
print("Testing rows:", len(test_df))

print("\nClient overlap:")
print(
    len(
        set(train_df["client_hash_id"])
        &
        set(test_df["client_hash_id"])
    )
)

Training clients: 24
Testing clients: 6

Training rows: 28907
Testing rows: 455

Client overlap:
0


## 3. Train + compare vs my baseline

I will train Logistic Regression using three February features:

- `impressions_feb`
- `clicks_feb`
- `avg_position_feb`

The target is `march_zero_click`.

The model probability will be used as the ranking score.

For comparison, I will apply the Week-4 baseline rule to the same held-out test rows and compare both approaches using **Precision@20 and Precision@50**.

These metrics match the ranking-style evaluation used earlier in the starter work. Higher precision means that a larger share of the highest-ranked items had the measured March zero-click outcome.

The baseline thresholds are learned from the training portion only before being applied to the test portion.

In [7]:
# model training

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

FEATURES = [
    "impressions_feb",
    "clicks_feb",
    "avg_position_feb",
]

X_train = train_df[FEATURES].copy()
y_train = train_df["march_zero_click"]

X_test = test_df[FEATURES].copy()
y_test = test_df["march_zero_click"]

model = Pipeline([
    ("scaler", StandardScaler()),
    (
        "logistic",
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    ),
])

model.fit(X_train, y_train)

test_df["model_score"] = model.predict_proba(X_test)[:, 1]

print("Model trained.")

Model trained.


In [8]:
# Reproduce baseline on the same test set
# Baseline thresholds learned from training data only.

volume_median = train_df["impressions_feb"].median()
volume_q75 = train_df["impressions_feb"].quantile(0.75)

position_median = train_df["avg_position_feb"].median()
position_q75 = train_df["avg_position_feb"].quantile(0.75)

print("Training-derived baseline thresholds:")
print(f"Volume median:     {volume_median:.2f}")
print(f"Volume 75th pct:   {volume_q75:.2f}")
print(f"Position median:   {position_median:.2f}")
print(f"Position 75th pct: {position_q75:.2f}")

Training-derived baseline thresholds:
Volume median:     2618.00
Volume 75th pct:   5463.50
Position median:   5.42
Position 75th pct: 8.75


In [9]:
# Baseline score
test_df["baseline_score"] = (
    np.where(
        test_df["impressions_feb"] >= volume_q75,
        2,
        np.where(
            test_df["impressions_feb"] >= volume_median,
            1,
            0
        )
    )
    +
    np.where(
        test_df["avg_position_feb"] >= position_q75,
        2,
        np.where(
            test_df["avg_position_feb"] >= position_median,
            1,
            0
        )
    )
)

print("Baseline scores:")
display(
    test_df["baseline_score"]
    .value_counts()
    .sort_index()
)

Baseline scores:


,count
baseline_score,
0,246
1,161
2,42
3,6


In [10]:
# Precision@K
def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))
    top_k = order[:k]
    return np.mean(np.asarray(y_true)[top_k])

for k in [20, 50]:
    model_p = precision_at_k(
        test_df["march_zero_click"].values,
        test_df["model_score"].values,
        k
    )

    baseline_p = precision_at_k(
        test_df["march_zero_click"].values,
        test_df["baseline_score"].values,
        k
    )

    print(f"Precision@{k}")
    print(f"Model:    {model_p:.3f}")
    print(f"Baseline: {baseline_p:.3f}")
    print()

Precision@20
Model:    0.250
Baseline: 0.050

Precision@50
Model:    0.260
Baseline: 0.080



In [11]:
# model-vs-baseline table
results = []

for k in [20, 50]:
    model_p = precision_at_k(
        test_df["march_zero_click"].values,
        test_df["model_score"].values,
        k
    )

    baseline_p = precision_at_k(
        test_df["march_zero_click"].values,
        test_df["baseline_score"].values,
        k
    )

    results.append({
        "metric": f"Precision@{k}",
        "baseline": baseline_p,
        "logistic_regression": model_p,
        "difference_model_minus_baseline": model_p - baseline_p,
    })

comparison = pd.DataFrame(results)

display(comparison)

,metric,baseline,logistic_regression,difference_model_minus_baseline
0,Precision@20,0.05,0.25,0.20
1,Precision@50,0.08,0.26,0.18


In [12]:
# Add a secondary classification metric
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)

roc_auc = roc_auc_score(
    y_test,
    test_df["model_score"]
)

avg_precision = average_precision_score(
    y_test,
    test_df["model_score"]
)

print(f"ROC-AUC: {roc_auc:.3f}")
print(f"Average Precision: {avg_precision:.3f}")

ROC-AUC: 0.825
Average Precision: 0.216


In [13]:
# Inspect model coefficients
coef = model.named_steps["logistic"].coef_[0]

coefficient_table = pd.DataFrame({
    "feature": FEATURES,
    "coefficient": coef,
    "absolute_coefficient": np.abs(coef),
}).sort_values(
    "absolute_coefficient",
    ascending=False
)

display(coefficient_table)

,feature,coefficient,absolute_coefficient
1,clicks_feb,-7.691205,7.691205
0,impressions_feb,-2.278688,2.278688
2,avg_position_feb,0.165532,0.165532


## 4. Errors and interpretation

I will inspect the highest-ranked predictions and the cases where the model and baseline disagree.

A high model score does not mean that the content definitely needs an action. It means that, based on the February features available to the model, the row received a higher predicted probability of the March zero-click outcome.

The error analysis will focus on false positives, false negatives, and disagreements between the learned model and the Week-4 heuristic.

In [14]:
# false positives / false negatives
test_df["model_pred"] = (
    test_df["model_score"] >= 0.5
).astype(int)

false_positives = test_df[
    (test_df["model_pred"] == 1)
    & (test_df["march_zero_click"] == 0)
].copy()

false_negatives = test_df[
    (test_df["model_pred"] == 0)
    & (test_df["march_zero_click"] == 1)
].copy()

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

False positives: 0
False negatives: 27


In [15]:
# inspect errors
error_columns = [
    "client_hash_id",
    "content_hash_id",
    "impressions_feb",
    "clicks_feb",
    "avg_position_feb",
    "march_zero_click",
    "model_score",
    "baseline_score",
]

print("False positives:")
display(
    false_positives
    .sort_values("model_score", ascending=False)
    [error_columns]
    .head(10)
)

print("False negatives:")
display(
    false_negatives
    .sort_values("model_score", ascending=True)
    [error_columns]
    .head(10)
)

False positives:


,client_hash_id,content_hash_id,impressions_feb,clicks_feb,avg_position_feb,march_zero_click,model_score,baseline_score


False negatives:


,client_hash_id,content_hash_id,impressions_feb,clicks_feb,avg_position_feb,march_zero_click,model_score,baseline_score
29026,client_b10cb2997d0c7c86,content_5771ae55202a3af9,990.0,9.0,3.854545,1,0.039299,0
14721,client_3ffa76342f366962,content_ce23841e74bfcd8b,111.0,8.0,5.144144,1,0.060156,0
14268,client_0fa64a184f18a4a0,content_d9923d1bcb6a74ab,575.0,6.0,1.553043,1,0.066293,0
29204,client_b10cb2997d0c7c86,content_ff7283bfa066e6bc,615.0,6.0,7.944715,1,0.075456,1
14657,client_3f0ce4d44fe94f3d,content_2f5787225ea68673,1138.0,4.0,6.358524,1,0.085344,1
29222,client_3f0ce4d44fe94f3d,content_b20b61f679a80a67,1071.0,4.0,7.179272,1,0.088427,1
14250,client_0fa64a184f18a4a0,content_b428e9155cf61eb4,601.0,4.0,2.510815,1,0.090577,0
14494,client_3ffa76342f366962,content_ea3f7498a05a3a6b,140.0,5.0,6.342857,1,0.095886,1
14240,client_0fa64a184f18a4a0,content_38e1a7edd7707634,316.0,4.0,2.806962,1,0.098255,0
3742,client_0fa64a184f18a4a0,content_2cd992440b27266b,337.0,4.0,3.071217,1,0.098267,0


In [16]:
# model vs baseline disagreements
# Convert baseline score into its prioritization order.
# Higher score = higher baseline priority.

test_df["baseline_rank"] = (
    test_df["baseline_score"]
    .rank(
        method="first",
        ascending=False
    )
)

test_df["model_rank"] = (
    test_df["model_score"]
    .rank(
        method="first",
        ascending=False
    )
)

disagreements = test_df.copy()

disagreements["rank_difference"] = (
    disagreements["baseline_rank"]
    - disagreements["model_rank"]
).abs()

display(
    disagreements
    .sort_values("rank_difference", ascending=False)
    [
        [
            "content_hash_id",
            "impressions_feb",
            "clicks_feb",
            "avg_position_feb",
            "march_zero_click",
            "baseline_score",
            "model_score",
            "baseline_rank",
            "model_rank",
            "rank_difference",
        ]
    ]
    .head(10)
)

,content_hash_id,impressions_feb,clicks_feb,avg_position_feb,march_zero_click,baseline_score,model_score,baseline_rank,model_rank,rank_difference
29201,content_e32ff455bb56bf8f,18089.0,86.0,5.750456,0,3,1.161772e-09,5.0,455.0,450.0
29017,content_4257d1459eb1fdd1,32934.0,4.0,6.748011,0,3,9.110059e-06,4.0,449.0,445.0
29207,content_bb21064343842c2f,7132.0,26.0,7.789961,0,3,4.815896e-04,6.0,438.0,432.0
14280,content_fbfecfbacfe7b2f2,6538.0,19.0,1.855307,0,2,1.541398e-03,10.0,430.0,420.0
14628,content_b2cb68466ab4baa4,8375.0,13.0,7.712716,0,3,2.734581e-03,1.0,420.0,419.0
29121,content_3bfd8d41b6e16051,284.0,3.0,4.147887,0,0,1.177305e-01,443.0,28.0,415.0
28763,content_073eeab0107ad398,3845.0,21.0,11.350325,0,3,3.049288e-03,3.0,418.0,415.0
14641,content_99e331f3466d8a9f,6159.0,27.0,3.907940,0,2,4.959930e-04,23.0,436.0,413.0
29102,content_2a85479c797b8fc5,223.0,3.0,3.896861,0,0,1.189623e-01,430.0,24.0,406.0
28761,content_382ce19bd9061566,5510.0,22.0,5.032668,0,2,1.379398e-03,28.0,431.0,403.0


### Interpretation

On the held-out client-grouped test set, the model's Precision@20 was **0.25** compared with **0.05** for the Week-4 baseline. At Precision@50, the model measured **0.26** compared with **0.08** for the baseline.

The model's coefficients indicate that it relies most strongly on **clicks_feb**, followed by **impressions_feb**. These coefficients describe the learned association in this dataset; they do not establish causation.

The error review shows that some high-scored rows are false positives because the February features do not capture all factors affecting the March outcome. False negatives show the opposite limitation: some March zero-click rows were not strongly signaled by the available February features.

The model and baseline therefore provide different decision-support rankings. Model complexity alone is not treated as an improvement; the comparison is based on the same held-out rows and the same ranking metrics.

In [17]:
# Final feature leakage check.

forbidden = [
    "march_zero_click",
    "clicks_mar",
    "available_days_mar",
    "trend_direction",
    "march_zero_click_rate",
    "future_clicks",
    "future_impressions",
]

used_features = FEATURES

leaked = [
    col for col in used_features
    if col in forbidden
]

print("Model features:", used_features)
print("Forbidden features found:", leaked)

assert not leaked, f"Leakage detected: {leaked}"

print("Feature leakage check: PASS")

Model features: ['impressions_feb', 'clicks_feb', 'avg_position_feb']
Forbidden features found: []
Feature leakage check: PASS


## Self-check

- [x] Every section is filled — markdown reasoning and supporting code are present.
- [x] The notebook runs top to bottom with no errors.
- [x] Logistic Regression is used as the primary model.
- [x] The target is the March 2026 zero-click outcome.
- [x] February features are used as the decision-time inputs.
- [x] Training and test clients do not overlap.
- [x] Baseline and model are evaluated on the same held-out test rows.
- [x] The baseline thresholds are learned from the training portion before evaluation.
- [x] Precision@20 and Precision@50 are reported for both the model and baseline.
- [x] Model errors and feature interpretation are inspected.
- [x] No March outcome information is included as a model feature.
- [x] No future-window or label-derived feature is used during training.
- [x] Claims use careful language: observed, measured, directional, decision-support.
- [x] No client names, URLs, or private queries are included.
- [x] The notebook is committed under `work/notebooks/w05_model.ipynb`.